# Synthetic-Depth Deployment Evaluation for `attention_fusion`

## Why this notebook

The production posture model ([`attention_fusion.pth`](../../../../models/vision-spatial/attention_fusion.pth)) reports test macro F1 **0.998** on SLP-2022 — but that test set uses **real depth from the same Kinect-class sensor** the model was trained on.

Production deployment likely runs on plain RGB video without a depth sensor. To run the model in that setting, the depth channel has to be **synthesized from RGB** with a monocular depth estimator. This introduces a domain shift not covered by the original test:

- Real depth = sensor-calibrated metric distance with characteristic noise (Kinect IR artifacts, edge bleeding, holes).
- Synthetic depth = a neural network's *guess* of relative depth from RGB alone — texture-based, often smooth where real depth has discontinuities.

**This notebook quantifies the deployment gap.** It runs the same `attention_fusion.pth` checkpoint twice on the same test split:

| Run | Depth source | RGB | Joints |
|---|---|---|---|
| **A** (baseline / sanity check) | Real Kinect depth from `depth/` | Same | Same |
| **B** (deployment) | Synthetic depth from Depth Anything v2 | Same | Same |

Goal: produce a concrete number for the F1 drop, and look at *which postures* degrade most. That measurement informs whether the late-fusion design can still treat vision posture as essentially-reliable, or whether it needs to be downgraded to event-annotation only with confidence gating.

## Outputs

- Cached synthetic depth PNGs at `experiments/artifacts/vision-spatial/inference_experiments/synthetic_depth_eval/depth_synth_cache/`
- Side-by-side metrics + confusion matrices at `experiments/artifacts/vision-spatial/inference_experiments/synthetic_depth_eval/results/`
- Final summary cell (Section 14) — the headline number for the writeup.

## Dependencies

Uses HuggingFace `transformers` (already in `requirements.txt`) to load Depth Anything v2 Small. First run downloads ~50 MB of weights to the HF cache.

## What stays the same as `attention_fusion_experiment.ipynb`

- Same dataset root, same metadata builder, same subject-wise split (`seed=42`, 70/15/15) — *identical test subjects and frames* as the original training run.
- Same encoder + fusion architecture.
- Same RGB transform (`Resize(224, 224)` + `ToTensor()`, no ImageNet normalization).
- Same joint normalization (x/576, y/1024).

## What changes

- A new `SLP_DATASET_ROOT` override (the original notebook hardcoded an absolute path that doesn't match this machine).
- The `FusionDataset` class accepts a `depth_source` toggle: `"real"` reads from `depth/`, `"synthetic"` reads from the cached synthesized PNGs.
- A precompute step generates and caches synthetic depth for every test-set frame.

## 1. Imports and setup

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import torchvision.transforms as T

from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Paths and configuration

In [ ]:
CWD = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [CWD, *CWD.parents] if (p / "requirements.txt").exists()), CWD)

# SLP dataset is one level above the project root in this environment.
SLP_DATASET_ROOT = PROJECT_ROOT.parent / "Dataset" / "SLP2022" / "SLP" / "danaLab"
CSV_PATH = SLP_DATASET_ROOT / "posture_labels_all_modalities.csv"

# Production checkpoint (same one referenced in the model card).
FUSION_CHECKPOINT = PROJECT_ROOT / "models" / "vision-spatial" / "attention_fusion.pth"

# Synthetic-depth eval artifacts (moved from fusion/attention_fusion/ to inference_experiments/)
EVAL_DIR = PROJECT_ROOT / "experiments" / "artifacts" / "vision-spatial" / "inference_experiments" / "synthetic_depth_eval"
DEPTH_SYNTH_CACHE = EVAL_DIR / "depth_synth_cache"
RESULTS_DIR = EVAL_DIR / "results"
DEPTH_SYNTH_CACHE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Image dimensions used to normalize 2D joint coordinates (matches training).
IMG_W = 576.0
IMG_H = 1024.0

CLASS_NAMES = ["supine", "left", "right"]
NUM_CLASSES = 3

BATCH_SIZE = 32
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

for label, p in [
    ("SLP root", SLP_DATASET_ROOT),
    ("Labels CSV", CSV_PATH),
    ("Fusion checkpoint", FUSION_CHECKPOINT),
    ("Eval artifacts", EVAL_DIR),
]:
    print(f"{label}: {p}")
    print(f"  exists: {p.exists()}")

## 3. Sanity check — real depth format

Before generating synthetic depth, verify what the model actually consumes. The fusion notebook loads depth via `Image.open(...).convert("L")` followed by `Resize((224, 224))` and `ToTensor()`. So the input to the encoder is a `(1, 224, 224)` tensor in `[0, 1]`.

We need our synthetic depth to follow the **same convention**: 8-bit grayscale PNG, where *brighter pixel = closer to camera* (verified by inspection of the raw NPY vs the PNG: raw mean 2207 mm → PNG mean ~187/255, consistent with `png = 255 × (1 − raw/max)`).

In [ ]:
sample_depth_path = SLP_DATASET_ROOT / "00001" / "depth" / "uncover" / "image_000001.png"
sample_rgb_path   = SLP_DATASET_ROOT / "00001" / "RGB"   / "uncover" / "image_000001.png"

depth_img = Image.open(sample_depth_path)
depth_arr = np.array(depth_img)
rgb_img = Image.open(sample_rgb_path).convert("RGB")

print("Real depth PNG")
print(f"  mode  : {depth_img.mode}")
print(f"  size  : {depth_img.size}  (PIL: width × height)")
print(f"  dtype : {depth_arr.dtype}")
print(f"  shape : {depth_arr.shape}")
print(f"  range : [{depth_arr.min()}, {depth_arr.max()}]")
print(f"  mean  : {depth_arr.mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(np.array(rgb_img))
axes[0].set_title("RGB")
axes[0].axis("off")
axes[1].imshow(depth_arr, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Real depth (closer = brighter)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 4. Reproduce the test split

Same `subject_wise_split` recipe + `seed=42` as `attention_fusion_experiment.ipynb`. This guarantees the test subjects and frames are identical to the original training/test run, so the synthetic-depth result is directly comparable to the 0.9981 baseline.

In [ ]:
def build_fusion_metadata(csv_path):
    df = pd.read_csv(csv_path)
    df["subject_id"] = df["subject_id"].astype(int).astype(str).str.zfill(5)
    rgb_df = df[df["modality"] == "RGB"].copy()
    rgb_df["rgb_path"] = (
        rgb_df["subject_id"] + "/RGB/" + rgb_df["condition"] +
        "/image_" + rgb_df["image_index"].astype(int).astype(str).str.zfill(6) + ".png"
    )
    rgb_df["depth_path"] = (
        rgb_df["subject_id"] + "/depth/" + rgb_df["condition"] +
        "/image_" + rgb_df["image_index"].astype(int).astype(str).str.zfill(6) + ".png"
    )
    rgb_df["joint_file"] = rgb_df["subject_id"] + "/joints_gt_RGB.mat"
    rgb_df["frame_idx_0based"] = rgb_df["image_index"] - 1
    return rgb_df[[
        "subject_id", "condition", "image_index",
        "label", "label_id",
        "rgb_path", "depth_path", "joint_file", "frame_idx_0based",
    ]].reset_index(drop=True)


def subject_wise_split(subject_ids, train_ratio=0.70, val_ratio=0.15, seed=42):
    subject_ids = sorted(subject_ids)
    rng = random.Random(seed)
    rng.shuffle(subject_ids)
    n = len(subject_ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    return (
        subject_ids[:n_train],
        subject_ids[n_train:n_train + n_val],
        subject_ids[n_train + n_val:],
    )


fusion_df = build_fusion_metadata(CSV_PATH)
subjects = sorted(fusion_df["subject_id"].unique().tolist())
train_subjects, val_subjects, test_subjects = subject_wise_split(subjects, 0.70, 0.15, SEED)

test_df = fusion_df[fusion_df["subject_id"].isin(test_subjects)].reset_index(drop=True)

print(f"Total subjects        : {len(subjects)}")
print(f"Test subjects         : {len(test_subjects)}  ({test_subjects[:5]} ...)")
print(f"Test samples (frames) : {len(test_df)}")
print("\nClass distribution in test set:")
print(test_df["label"].value_counts().rename("count"))

## 5. Set up Depth Anything v2 (Small)

Depth Anything v2 is the current SOTA monocular depth estimator (CVPR 2024). The Small variant has ~25 M params and runs comfortably on consumer GPUs. It outputs **inverse depth** (closer = larger value) in arbitrary units; we normalize per-image to `[0, 255]` uint8 to match SLP's PNG convention.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

DEPTH_MODEL_NAME = "depth-anything/Depth-Anything-V2-Small-hf"

depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_NAME)
depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_NAME).to(device).eval()

n_params = sum(p.numel() for p in depth_model.parameters())
print(f"Depth Anything v2 Small loaded — {n_params/1e6:.1f}M params on {device}")

In [ ]:
@torch.no_grad()
def synthesize_depth_pil(rgb_pil, target_size=(424, 512)):
    """Run Depth Anything v2 on a PIL RGB image, return a PIL grayscale (L) PNG
    matching SLP's depth convention (uint8, closer=brighter, size=target_size).

    target_size is (width, height) — matches SLP's 424×512 depth PNGs.
    """
    inputs = depth_processor(images=rgb_pil, return_tensors="pt").to(device)
    outputs = depth_model(**inputs)
    pred = outputs.predicted_depth  # (1, h, w), inverse depth (close = larger)

    # Resize to target spatial size (PIL convention: width, height).
    pred = torch.nn.functional.interpolate(
        pred.unsqueeze(1),
        size=(target_size[1], target_size[0]),  # (H, W)
        mode="bicubic",
        align_corners=False,
    ).squeeze().cpu().numpy()

    # Per-image min-max normalize to [0, 255] uint8.
    # closer=larger inverse-depth aligns with closer=brighter PNG → no flip needed.
    p_min, p_max = pred.min(), pred.max()
    if p_max > p_min:
        norm = (pred - p_min) / (p_max - p_min)
    else:
        norm = np.zeros_like(pred)
    return Image.fromarray((norm * 255).astype(np.uint8), mode="L")


# Sanity check on the same sample we inspected in Section 3
synth_depth = synthesize_depth_pil(rgb_img, target_size=(424, 512))
synth_arr = np.array(synth_depth)

print("Synthetic depth")
print(f"  size  : {synth_depth.size}")
print(f"  range : [{synth_arr.min()}, {synth_arr.max()}]")
print(f"  mean  : {synth_arr.mean():.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(np.array(rgb_img));  axes[0].set_title("RGB");                   axes[0].axis("off")
axes[1].imshow(depth_arr, cmap="gray", vmin=0, vmax=255); axes[1].set_title("Real depth");  axes[1].axis("off")
axes[2].imshow(synth_arr, cmap="gray", vmin=0, vmax=255); axes[2].set_title("Synthetic depth"); axes[2].axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sample_depth_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Precompute synthetic depth for the test set

Generate synthetic depth for every test-set RGB image and cache as PNG. The cache is keyed by `(subject_id, condition, image_index)` — same triple as the real depth path. Subsequent runs skip already-generated files.

In [ ]:
n_total = len(test_df)
n_skipped = 0
n_computed = 0

for i, row in test_df.iterrows():
    out_dir = DEPTH_SYNTH_CACHE / row["subject_id"] / "depth" / row["condition"]
    out_path = out_dir / f"image_{int(row['image_index']):06d}.png"

    if out_path.exists():
        n_skipped += 1
        continue

    rgb_p = SLP_DATASET_ROOT / row["rgb_path"]
    rgb = Image.open(rgb_p).convert("RGB")
    synth = synthesize_depth_pil(rgb, target_size=(424, 512))
    out_dir.mkdir(parents=True, exist_ok=True)
    synth.save(out_path)
    n_computed += 1

    if (i + 1) % 100 == 0 or (i + 1) == n_total:
        print(f"  {i + 1}/{n_total}  computed={n_computed}  skipped={n_skipped}")

print(f"\nDone. computed={n_computed} | skipped (cache hit)={n_skipped} | total={n_total}")
print(f"Cache: {DEPTH_SYNTH_CACHE}")

## 7. Recreate the model and load production weights

The encoder + fusion classes are copied verbatim from `attention_fusion_experiment.ipynb` — same definitions used at training time, so `load_state_dict` will match key-for-key.

The checkpoint is the **production** copy at `models/vision-spatial/attention_fusion.pth` (the frozen-encoder version, not the rejected finetuned variant).

In [ ]:
# Encoder dimensions — match training
DEPTH_FEATURE_DIM = 512
RGB_FEATURE_DIM = 128
JOINT_INPUT_DIM = 42
JOINT_FEATURE_DIM = 128
COMMON_FEATURE_DIM = 256


class DepthEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        layers = list(backbone.children())[:-1]
        self.encoder = nn.Sequential(*layers)
        self.flatten = nn.Flatten()
        self.feature_dim = 512

    def forward(self, x):
        return self.flatten(self.encoder(x))


class RGBEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=None)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.projection = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 128),
        )
        self.feature_dim = 128

    def forward(self, x):
        return self.projection(self.backbone(x))


class JointEncoder(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=128, feature_dim=128, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, feature_dim), nn.ReLU(), nn.Dropout(dropout),
        )
        self.feature_dim = feature_dim

    def forward(self, x):
        return self.encoder(x)


class AttentionFusionClassifier(nn.Module):
    def __init__(
        self,
        depth_encoder, rgb_encoder, joint_encoder,
        depth_feature_dim=512, rgb_feature_dim=128, joint_feature_dim=128,
        common_feature_dim=256, num_heads=4, ff_dim=512, dropout=0.1,
        num_classes=3,
    ):
        super().__init__()
        self.depth_encoder = depth_encoder
        self.rgb_encoder = rgb_encoder
        self.joint_encoder = joint_encoder

        self.depth_proj = nn.Linear(depth_feature_dim, common_feature_dim)
        self.rgb_proj   = nn.Linear(rgb_feature_dim,   common_feature_dim)
        self.joint_proj = nn.Linear(joint_feature_dim, common_feature_dim)

        self.modality_embed = nn.Parameter(torch.randn(3, common_feature_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=common_feature_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.attention_block = nn.TransformerEncoder(encoder_layer, num_layers=1)

        self.classifier = nn.Sequential(
            nn.Linear(common_feature_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, depth_x, rgb_x, joint_x):
        with torch.no_grad():
            f_depth = self.depth_encoder(depth_x)
            f_rgb   = self.rgb_encoder(rgb_x)
            f_joint = self.joint_encoder(joint_x)
        t_depth = self.depth_proj(f_depth)
        t_rgb   = self.rgb_proj(f_rgb)
        t_joint = self.joint_proj(f_joint)
        tokens = torch.stack([t_rgb, t_depth, t_joint], dim=1)
        tokens = tokens + self.modality_embed.unsqueeze(0)
        tokens = self.attention_block(tokens)
        fused = tokens.mean(dim=1)
        return self.classifier(fused)


depth_encoder = DepthEncoder().to(device)
rgb_encoder   = RGBEncoder().to(device)
joint_encoder = JointEncoder(input_dim=JOINT_INPUT_DIM,
                              hidden_dim=128,
                              feature_dim=JOINT_FEATURE_DIM,
                              dropout=0.3).to(device)

fusion_model = AttentionFusionClassifier(
    depth_encoder=depth_encoder, rgb_encoder=rgb_encoder, joint_encoder=joint_encoder,
    depth_feature_dim=DEPTH_FEATURE_DIM, rgb_feature_dim=RGB_FEATURE_DIM,
    joint_feature_dim=JOINT_FEATURE_DIM, common_feature_dim=COMMON_FEATURE_DIM,
    num_heads=4, ff_dim=512, dropout=0.1, num_classes=NUM_CLASSES,
).to(device)

ckpt = torch.load(FUSION_CHECKPOINT, map_location=device, weights_only=False)
fusion_model.load_state_dict(ckpt["model_state_dict"])
fusion_model.eval()

n_params = sum(p.numel() for p in fusion_model.parameters())
print(f"Loaded {FUSION_CHECKPOINT.name} ({n_params/1e6:.1f}M params)")
print(f"Best val F1 (training): {ckpt.get('best_val_f1', 'unknown')}")

## 8. Dataset with toggleable depth source

The `FusionDataset` is identical to the training notebook except that `depth_source` chooses between the real Kinect depth (`depth/...` under the SLP root) and the cached synthetic depth (`depth_synth_cache/...` under the eval artifacts dir). Everything else — RGB, joints, transforms, label — is shared, so the only thing varying between Run A and Run B is the depth tensor.

In [ ]:
rgb_transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])
depth_transform = T.Compose([T.Resize((224, 224)), T.ToTensor()])


def preprocess_joint_frame_xyo(frame_joints):
    x = frame_joints[0].astype(np.float32) / IMG_W
    y = frame_joints[1].astype(np.float32) / IMG_H
    occ = frame_joints[2].astype(np.float32)
    return np.stack([x, y, occ], axis=1).reshape(-1)


class FusionDataset(Dataset):
    """Same as the training notebook, with a depth_source switch:
        - 'real'      : SLP_DATASET_ROOT / row['depth_path']
        - 'synthetic' : DEPTH_SYNTH_CACHE / {subject}/depth/{condition}/image_{idx}.png
    """
    def __init__(self, df, slp_root, depth_synth_root, depth_source="real",
                 rgb_tf=None, depth_tf=None):
        assert depth_source in {"real", "synthetic"}
        self.df = df.reset_index(drop=True)
        self.slp_root = Path(slp_root)
        self.depth_synth_root = Path(depth_synth_root)
        self.depth_source = depth_source
        self.rgb_tf = rgb_tf
        self.depth_tf = depth_tf
        self.joint_cache = {}

    def __len__(self):
        return len(self.df)

    def _load_joints(self, joint_rel_path):
        if joint_rel_path not in self.joint_cache:
            self.joint_cache[joint_rel_path] = sio.loadmat(self.slp_root / joint_rel_path)["joints_gt"]
        return self.joint_cache[joint_rel_path]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rgb = Image.open(self.slp_root / row["rgb_path"]).convert("RGB")
        if self.rgb_tf:
            rgb = self.rgb_tf(rgb)

        if self.depth_source == "real":
            depth_path = self.slp_root / row["depth_path"]
        else:
            depth_path = (
                self.depth_synth_root
                / row["subject_id"] / "depth" / row["condition"]
                / f"image_{int(row['image_index']):06d}.png"
            )
        depth = Image.open(depth_path).convert("L")
        if self.depth_tf:
            depth = self.depth_tf(depth)

        joints_all = self._load_joints(row["joint_file"])
        frame = joints_all[:, :, int(row["frame_idx_0based"])]
        joint_vec = preprocess_joint_frame_xyo(frame)

        return {
            "rgb": rgb,
            "depth": depth,
            "joint": torch.tensor(joint_vec, dtype=torch.float32),
            "label": torch.tensor(int(row["label_id"]), dtype=torch.long),
        }


@torch.no_grad()
def evaluate(model, loader, device, label=""):
    model.eval()
    y_true, y_pred = [], []
    for i, batch in enumerate(loader):
        d = batch["depth"].to(device, non_blocking=True)
        r = batch["rgb"].to(device, non_blocking=True)
        j = batch["joint"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)
        logits = model(d, r, j)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        if (i + 1) % 10 == 0:
            print(f"  {label} batch {i + 1} / ~{len(loader)}")
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    return {
        "y_true": y_true,
        "y_pred": y_pred,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "per_class": classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                            output_dict=True, zero_division=0),
        "confusion": confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES))),
    }

## 9. Run A — real depth (sanity check)

This run should reproduce the **0.9981** macro-F1 baseline from the original notebook. If it doesn't, something has drifted in the dataset / split / model definitions and the synthetic-depth comparison would be untrustworthy.

In [ ]:
test_real = FusionDataset(
    test_df, SLP_DATASET_ROOT, DEPTH_SYNTH_CACHE,
    depth_source="real",
    rgb_tf=rgb_transform, depth_tf=depth_transform,
)
loader_real = DataLoader(test_real, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

results_real = evaluate(fusion_model, loader_real, device, label="REAL")

print("\n=== Run A — REAL depth ===")
print(f"Accuracy : {results_real['accuracy']:.4f}")
print(f"Macro F1 : {results_real['macro_f1']:.4f}")

## 10. Run B — synthetic depth (deployment)

In [ ]:
test_synth = FusionDataset(
    test_df, SLP_DATASET_ROOT, DEPTH_SYNTH_CACHE,
    depth_source="synthetic",
    rgb_tf=rgb_transform, depth_tf=depth_transform,
)
loader_synth = DataLoader(test_synth, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

results_synth = evaluate(fusion_model, loader_synth, device, label="SYNTH")

print("\n=== Run B — SYNTHETIC depth ===")
print(f"Accuracy : {results_synth['accuracy']:.4f}")
print(f"Macro F1 : {results_synth['macro_f1']:.4f}")

## 11. Side-by-side comparison

Per-class precision / recall / F1 + confusion matrices.

In [ ]:
rows = []
for cls in CLASS_NAMES:
    real = results_real["per_class"][cls]
    synth = results_synth["per_class"][cls]
    rows.append({
        "class": cls,
        "support": int(real["support"]),
        "precision_real": real["precision"], "precision_synth": synth["precision"], "Δprecision": synth["precision"] - real["precision"],
        "recall_real":    real["recall"],    "recall_synth":    synth["recall"],    "Δrecall":    synth["recall"]    - real["recall"],
        "f1_real":        real["f1-score"],  "f1_synth":        synth["f1-score"],  "Δf1":        synth["f1-score"]  - real["f1-score"],
    })
comparison_df = pd.DataFrame(rows)
print("Per-class precision / recall / F1 — real vs synthetic")
print(comparison_df.round(4).to_string(index=False))

print("\nHeadline")
print(f"  Real depth      : acc {results_real['accuracy']:.4f}  | macro F1 {results_real['macro_f1']:.4f}")
print(f"  Synthetic depth : acc {results_synth['accuracy']:.4f}  | macro F1 {results_synth['macro_f1']:.4f}")
print(f"  Δ               : acc {results_synth['accuracy'] - results_real['accuracy']:+.4f}  | macro F1 {results_synth['macro_f1'] - results_real['macro_f1']:+.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (cm, title) in zip(axes, [
    (results_real["confusion"],  f"Real depth — F1 {results_real['macro_f1']:.4f}"),
    (results_synth["confusion"], f"Synthetic depth — F1 {results_synth['macro_f1']:.4f}"),
]):
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.suptitle("attention_fusion — real vs synthetic depth on the SLP-2022 test split", fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Where do the two runs disagree?

Sample-level breakdown of the *additional* errors caused by switching to synthetic depth — i.e., samples that real-depth got right but synthetic-depth got wrong. This identifies the failure modes specific to synthetic depth.

In [ ]:
y_true        = results_real["y_true"]
preds_real    = results_real["y_pred"]
preds_synth   = results_synth["y_pred"]

real_correct  = preds_real  == y_true
synth_correct = preds_synth == y_true
regressed     = real_correct & ~synth_correct           # real got it, synth didn't
recovered     = ~real_correct & synth_correct           # synth got it, real didn't
both_wrong    = ~real_correct & ~synth_correct
agree         = preds_real == preds_synth

print("Sample-level breakdown")
print(f"  Total samples       : {len(y_true)}")
print(f"  Both correct        : {(real_correct & synth_correct).sum()}")
print(f"  Both wrong          : {both_wrong.sum()}")
print(f"  Regressed (real→synth wrong): {regressed.sum()}")
print(f"  Recovered (synth flipped right): {recovered.sum()}")
print(f"  Predictions agree   : {agree.sum()} ({agree.mean()*100:.1f}%)")

if regressed.sum() > 0:
    print("\nRegressions — true-class → synth-predicted-class breakdown:")
    reg_df = pd.DataFrame({
        "true":       [CLASS_NAMES[c] for c in y_true[regressed]],
        "synth_pred": [CLASS_NAMES[c] for c in preds_synth[regressed]],
    })
    print(reg_df.groupby(["true", "synth_pred"]).size().rename("count").to_frame())

## 13. Save artifacts

In [ ]:
summary = {
    "experiment": "attention_fusion_synthetic_depth_eval",
    "baseline_checkpoint": str(FUSION_CHECKPOINT.relative_to(PROJECT_ROOT)),
    "depth_estimator": DEPTH_MODEL_NAME,
    "test_samples": int(len(y_true)),
    "class_names": CLASS_NAMES,
    "real_depth": {
        "accuracy": results_real["accuracy"],
        "macro_f1": results_real["macro_f1"],
        "classification_report": results_real["per_class"],
        "confusion_matrix": results_real["confusion"].tolist(),
    },
    "synthetic_depth": {
        "accuracy": results_synth["accuracy"],
        "macro_f1": results_synth["macro_f1"],
        "classification_report": results_synth["per_class"],
        "confusion_matrix": results_synth["confusion"].tolist(),
    },
    "delta": {
        "accuracy": results_synth["accuracy"] - results_real["accuracy"],
        "macro_f1": results_synth["macro_f1"] - results_real["macro_f1"],
    },
    "sample_breakdown": {
        "both_correct":  int((real_correct & synth_correct).sum()),
        "both_wrong":    int(both_wrong.sum()),
        "regressed":     int(regressed.sum()),
        "recovered":     int(recovered.sum()),
        "agree_count":   int(agree.sum()),
        "agree_fraction": float(agree.mean()),
    },
    "test_subjects": sorted(test_subjects),
}

with open(RESULTS_DIR / "synthetic_depth_eval_results.json", "w") as f:
    json.dump(summary, f, indent=2)

comparison_df.round(4).to_csv(RESULTS_DIR / "per_class_comparison.csv", index=False)

print("Saved:")
for p in sorted(RESULTS_DIR.glob("*")):
    print(f"  {p.relative_to(PROJECT_ROOT)}")

## 14. Findings

### Headline

| Metric | Real depth (training domain) | Synthetic depth (deployment) | Δ |
|---|---|---|---|
| Test accuracy | 0.9981 | **0.8806** | **−0.1176** |
| Test macro F1 | 0.9981 | **0.8751** | **−0.1230** |

The Run A baseline reproduces the original 0.9981 number exactly — confirms the eval pipeline is correct, the comparison is apples-to-apples. The deployment gap is **about 12 percentage points** in both accuracy and macro F1.

### Per-class breakdown — the critical pattern

| Class | F1 Real | F1 Synth | ΔF1 | What dropped |
|---|---|---|---|---|
| **supine** | 0.997 | **0.783** | **−0.21** | Recall collapsed from 0.999 → **0.643** (precision actually went up) |
| left | 0.997 | 0.981 | −0.02 | Almost unaffected |
| **right** | 1.000 | **0.862** | −0.14 | Precision collapsed from 1.000 → **0.757** (recall stayed at 1.000) |

These two patterns are the same failure expressed two ways: **the model is misclassifying supine as right when given synthetic depth**. From the disagreement analysis (Section 12):

- 256 frames regressed (real-depth correct → synth-depth wrong) out of 2,160 = **11.8% of test set**
- Of those 256 regressions: **230 are true-supine predicted as right**, 26 are true-supine predicted as left
- All other class transitions are negligible

So the deployment failure is concentrated in **one specific confusion: supine → right**. This is the most clinically important class to get right (supine OSA is the most common and most actionable apnea presentation), and it's exactly where synthetic depth fails hardest.

### Why this happens (interpretation)

- Real Kinect depth captures lateral asymmetry of the body even through blankets — supine is bilaterally symmetric, side postures show clear depth gradient.
- Monocular depth from RGB cannot recover this 3D structure when the body is occluded by bedding. The depth model essentially predicts a flat-ish surface for all postures, and the encoder loses its main supine-vs-lateral cue.
- The model still gets *some* class signal from RGB and joints — that's why F1 is 0.875, not 0.5 — but the depth modality has gone from informative to actively misleading for supine.

### What this means for late fusion

The **asymmetric design** (vision = reliable, modifies audio threshold) is no longer justified at deployment. Concretely:

- Vision macro F1 in deployment ≈ **0.875**, not 0.998. Only marginally better than audio's 0.777.
- **Supine recall = 0.643** in deployment. If we used vision posture to *lower* the audio apnea threshold for supine cases (the original Layer-3 plan), we would apply that lower threshold to only ~64% of true supine frames — and the missed 36% would not get the sensitivity boost they need. False posture calls (e.g., supine → "right") would push apnea-prone segments to the *higher* threshold, **suppressing exactly the events we'd want to catch most**.
- Annotating events with posture is still useful — even at 87.5% accuracy, that's signal — but the annotation itself should be flagged when vision is uncertain.

### Updated late-fusion design

| Layer | Strategy | Status |
|---|---|---|
| 1 | Audio hysteresis (≥2 consecutive windows) | **Keep** |
| 2 | Audio confidence gating | **Keep** |
| 3 | Posture aggregation per audio window | **Keep** |
| 4 | ~~Posture-conditioned audio threshold~~ | **DROP** — vision not reliable enough at deployment |
| 5 | Posture as event annotation | **Keep**, but treat as advisory, not decision-input |
| 6 | **Vision confidence gating** (new — was optional, now mandatory) | **Add** — flag `posture: uncertain` when max vision softmax < threshold |

The system still emits two timelines + annotated events, but the apnea decision comes solely from audio. Posture is metadata that the consumer (clinician / downstream analytics) can use, with an explicit confidence flag when vision is unsure.

### What to write up

> "We evaluated the deployment gap for the vision-spatial model by replacing real Kinect depth with synthetic depth from Depth Anything v2 on the held-out SLP-2022 test split. Macro F1 dropped from 0.998 to 0.875 (Δ = −0.123), with the regression concentrated in supine recall (0.999 → 0.643). The failure mode is monocular depth's inability to recover lateral body asymmetry when the subject is covered by bedding — it predicts a near-flat surface, and the encoder loses its primary supine-vs-side cue.
>
> Given this gap, the late-fusion design treats vision posture as **event annotation only**, not as a modifier of the audio apnea threshold. A posture-conditioned threshold strategy remains a clear future-work direction once either (a) a deployment dataset with real depth is available, or (b) the vision model is retrained with synthetic depth as input."

This frames the result as a concrete, measured limitation that **drives** the fusion design choice, not as a generic caveat.

### Open path if you want to recover the gap

Two future-work options that would close some or all of the deployment gap:

1. **Augment training with synthetic depth.** Generate Depth Anything v2 outputs on SLP-2022 training images, train the encoder on a mix of real and synthetic depth (or just synthetic). Brings deployment in-distribution. Would need a retraining pass (~30 min on GPU based on the original notebook).
2. **Retrain the vision model with depth dropped.** Use only RGB + joints as input. Loses the multimodal benefit of depth entirely, but eliminates the synthetic-depth domain-shift problem. Likely lands somewhere between 0.875 and 0.998 — could be better or worse than synthetic-depth deployment depending on how much depth contributed.

Both are out of scope for the current commit but worth flagging in the writeup as the next investigations.